# Nepali Extractive QA LoRA Fine-tune v4 — Llama 3.2 3B Instruct, Unsloth, Kaggle

> **Update (v4, this version):** switched the synthetic set to the cleaned v2 dataset, **`iwasbinod/QA_nepali_5000_pairs_syn`** (~5,000 pairs, was `QA_nepali_syn_jsonl_format` ~4,000). Hyperparameters pulled back to a safer config for clean data: LoRA `r` 32→16, `alpha` 64→32, `learning_rate` 5e-5→3e-5, `max_seq_length` 896→768. Added a **`SYNTHETIC_ONLY`** toggle (default `True`) that trains on just the cleaned v2 set and skips the ~60-70 min SQuAD-ne translation stage entirely — set it `False` to mix SQuAD-ne back in (~50/50 with the synthetic set) for extra training signal. Also fixed a real scoring bug visible in the last run's sample outputs: the model sometimes emitted literal `<|im_end|>` (or a lookalike token) instead of stopping cleanly, which was silently corrupting F1 — predictions are now trimmed at the first `<|` regardless of exact spelling. Adapter now saved/pushed as **`llama-3.2-3b_qa_syn_v2`**, cached dataset as **`iwasbinod/nepsquad-ne-v2`** (renamed so this run can't silently overwrite, or be confused with, the earlier v1 run).

> **Update (v3):** added **Dataset 4** — your synthetic Nepali QA set (`iwasbinod/QA_nepali_syn_jsonl_format`, ~4,000 pairs), span-aligned to the same "answer copied from context" convention as SQuAD-ne and de-duplicated against it (see section 6b). The adapter was saved/pushed as `llama-3.2-3b_qa_syn_updated_v1` (was `llama-3.2-3b-nepali-qa-lora-v1`). Final eval prints a clear pass/fail verdict against the base model.

**Why this run should beat the last one:** the v1 run (F1 25.21 vs base 26.69) ran on a noisier synthetic set with more LoRA capacity and a higher LR than clean data needs — that combination makes it easy to drift away from the base model's existing knowledge without actually learning the extraction task better. The v2 dataset is cleaned, LoRA capacity/LR are both pulled back so there's less room to overwrite base knowledge on a mid-size dataset, and the token-stripping fix removes a scoring artifact that was dragging F1 down independent of model quality.

**Budget: with `SYNTHETIC_ONLY=True` (default), ~50-70 min total** — no translation stage needed. Set `SYNTHETIC_ONLY=False` to also translate + mix in SQuAD-ne, which brings the budget back up to roughly the time budget described below.

Recommended config for this run (v2 cleaned dataset), safer and better suited to clean extractive data than the previous run:

| Parameter | Value | Why |
|---|---|---|
| LoRA rank (`r`) | 16 | Lower rank = less risk of overfitting / forgetting base knowledge |
| LoRA alpha | 32 | Kept at 2 × r |
| LoRA dropout | 0.05 | Unchanged |
| Learning rate | 3e-5 | Safer than 5e-5/1e-4 — clean data needs less aggressive LR |
| Epochs | 2 | Start here; only raise to 3 if final training loss is still high |
| `max_seq_length` | 768 | Safer on a T4 than 896 (bump back up if memory allows) |
| `per_device_train_batch_size` | 1 | Unchanged |
| `gradient_accumulation_steps` | 16 | Effective batch size 16 |
| `warmup_steps` | 8% of total steps | Unchanged |
| optimizer | `adamw_8bit` | Unchanged |
| `lr_scheduler` | `cosine` | Unchanged |
| `weight_decay` | 0.01 | Unchanged |
| `packing` | `False` | Must stay `False` for extractive QA |

Training data:
- **Dataset 4 (main, default)**: your cleaned synthetic set, `iwasbinod/QA_nepali_5000_pairs_syn` (~1,000 pairs), span-aligned against its own context (section 6b).i had used 1k high quality dataset synthetic not other, it is : iwasbinod/nepali_qa_latest_updated_1k_high_qualty_dataset_s-09aug

- 
- **SQuAD v1.1 → Nepali** (machine-translated + fuzzy-aligned, ~2500-3000 examples after alignment) — only built when `SYNTHETIC_ONLY=False`, as extra extractive-QA supervision.
- **`ai4bharat/IndicQA` Hindi subset** — still excluded (see cell 5). Kept out deliberately: you asked for a pure-Nepali result, and mixing in Hindi risks phrasing leakage since it's the same LoRA weights producing both languages.
- **Hard negatives**: off by default (`N_NEGATIVES = 0`) — earlier runs showed this maximizes F1 on the all-answerable Yunika eval set.

**Held out entirely for evaluation**: `Yunika/Nepali-QA` — all 266 examples, untouched, human-written, all-answerable. The final cell reports the fine-tuned model's refusal rate on this all-answerable set as a direct check that hard-negative data isn't quietly costing you F1 (moot while `N_NEGATIVES=0`, but kept as a safety check).

not 5000 -- its 1k

## 0. GPU check

In [1]:
# cell 0
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU DETECTED - enable a GPU accelerator in the Kaggle notebook settings panel."


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


## 1. Install dependencies

In [2]:
# cell 1
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # must be set before torch/CUDA init to take effect

!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q trl peft accelerate bitsandbytes datasets huggingface_hub sentencepiece rapidfuzz

import torch, time, re, gc, json
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset, concatenate_datasets
import pandas as pd
from huggingface_hub import login

t_notebook_start = time.time()
print("Setup complete.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 262.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 374.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 275.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 258.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 362.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 347.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 326.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 236.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 294.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 362.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 307.9 MB/

## 1b. Config — data source for this run (v2 cleaned dataset)

`SYNTHETIC_ONLY` controls whether sections 2-4 (NLLB load + SQuAD→Nepali translation/alignment)
run at all:

- **`True` (default, recommended first)** — train on just the cleaned `iwasbinod/QA_nepali_5000_pairs_syn`
  set. Sections 2-4 print a skip message and do nothing; `squad_ne_df` is created empty so the rest
  of the pipeline (combine/cache/train) runs unchanged. Fastest path, and the safest one to check
  whether the cleaned data alone beats base.
- **`False`** — also translate + mix in SQuAD-ne (~50/50 with the synthetic set) for extra training
  signal, at the cost of the ~60-70 min translation stage running again. Try this if the
  `SYNTHETIC_ONLY=True` run doesn't clearly beat base.

In [3]:
# ============================================================
# TRAINING DATA CONFIG - v2 cleaned dataset run
# ============================================================
SYNTHETIC_ONLY = False
# True  (default): train only on the cleaned iwasbinod/QA_nepali_5000_pairs_syn set (section 6b).
#        Skips NLLB load + SQuAD-ne translation/alignment (sections 2-4) entirely - saves ~60-70 min.
# False: also translate + mix in SQuAD-ne (~50/50 with the synthetic set) for extra training signal.

if SYNTHETIC_ONLY:
    print("SYNTHETIC_ONLY=True: training on the cleaned v2 synthetic set only. "
          "SQuAD-ne translation stages (sections 2-4) will be skipped below.")
else:
    print("SYNTHETIC_ONLY=False: will translate + mix in SQuAD-ne alongside the v2 synthetic set.")


SYNTHETIC_ONLY=False: will translate + mix in SQuAD-ne alongside the v2 synthetic set.


## 2. Translation engine (NLLB-200) — load once, reuse for both SQuAD and hard-negative translation

Skipped when `SYNTHETIC_ONLY=True` (the default for this run) since there's nothing to translate.

In [4]:
# cell 2
if not SYNTHETIC_ONLY:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    NLLB_MODEL = "facebook/nllb-200-distilled-600M"
    nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL, src_lang="eng_Latn")
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL, torch_dtype=torch.float16).to("cuda")
    nllb_model.eval()

    try:
        NPI_TOKEN_ID = nllb_tokenizer.convert_tokens_to_ids("npi_Deva")
    except Exception:
        NPI_TOKEN_ID = nllb_tokenizer.lang_code_to_id["npi_Deva"]

    @torch.no_grad()
    def nllb_translate_batch(texts, max_length=400, batch_size=32, log_every=20):
        outputs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = nllb_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to("cuda")
            gen = nllb_model.generate(
                **inputs, forced_bos_token_id=NPI_TOKEN_ID, max_length=max_length, num_beams=1, do_sample=False,
            )
            outputs.extend(nllb_tokenizer.batch_decode(gen, skip_special_tokens=True))
            if (i // batch_size) % log_every == 0:
                print(f"  translated {i+len(batch)}/{len(texts)}")
        return outputs

    print("NLLB-200 loaded.")
else:
    print("NLLB-200 load skipped (SYNTHETIC_ONLY=True).")


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB-200 loaded.


## 3. Fuzzy span alignment (shared by SQuAD-main and hard-negative translation)

In [5]:
# cell 3
from rapidfuzz import fuzz

MIN_MATCH_SCORE = 68  # raised from 62: stricter alignment means fewer but cleaner training labels -
                        # label quality matters more than raw count for teaching correct span extraction

def find_best_span(context_ne, answer_ne):
    ctx_words = context_ne.split()
    ans_len = max(len(answer_ne.split()), 1)
    best_score, best_span = -1, None
    for win in range(max(1, ans_len - 1), ans_len + 3):
        for i in range(0, max(1, len(ctx_words) - win + 1)):
            candidate = " ".join(ctx_words[i:i+win])
            score = fuzz.ratio(candidate, answer_ne)
            if score > best_score:
                best_score, best_span = score, candidate
    return best_span, best_score


## 4. Dataset 1 — SQuAD v1.1 → Nepali (main supervision, ~5000 examples)

Skipped entirely when `SYNTHETIC_ONLY=True` (the default for this run) — `squad_ne_df` is created empty instead so the rest of the pipeline runs unchanged.

In [6]:
# cell 4a - sample + dedupe by passage, pre-filtered by length to avoid wasted translation on passages
# that will just get dropped by the token-length filter later (Devanagari inflates token count a lot)
if not SYNTHETIC_ONLY:
    raw_squad = load_dataset("rajpurkar/squad", split="train")
    df_squad = raw_squad.to_pandas()

    context_lengths = df_squad["context"].str.len()
    length_ok = df_squad[(context_lengths >= 150) & (context_lengths <= 550)]  # short/medium passages translate
                                                                                  # to Nepali well within budget
    unique_contexts = length_ok["context"].drop_duplicates().reset_index(drop=True)
    print(f"Unique passages in the 150-550 char band: {len(unique_contexts)}")

    N_CONTEXTS = 1900
    MAX_Q_PER_CONTEXT = 2
    SEED = 42

    sampled_contexts = unique_contexts.sample(n=min(N_CONTEXTS, len(unique_contexts)), random_state=SEED).tolist()
    sampled_set = set(sampled_contexts)

    subset = df_squad[df_squad["context"].isin(sampled_set)].groupby("context").head(MAX_Q_PER_CONTEXT).reset_index(drop=True)
    subset["answer_text"] = subset["answers"].apply(lambda a: a["text"][0] if len(a["text"]) else "")
    subset = subset[subset["answer_text"].str.len() > 0].reset_index(drop=True)
    subset = subset.sample(n=min(3800, len(subset)), random_state=SEED).reset_index(drop=True)
    print(f"SQuAD QA pairs queued for translation: {len(subset)} across {subset['context'].nunique()} passages")
else:
    print("SQuAD sampling skipped (SYNTHETIC_ONLY=True).")


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Unique passages in the 150-550 char band: 4865
SQuAD QA pairs queued for translation: 3710 across 1900 passages


In [7]:
# cell 4b - translate + align
if not SYNTHETIC_ONLY:
    t0 = time.time()
    unique_ctx_list = subset["context"].drop_duplicates().tolist()
    ctx_map = dict(zip(unique_ctx_list, nllb_translate_batch(unique_ctx_list, max_length=400)))
    subset["context_ne"] = subset["context"].map(ctx_map)
    print(f"Passages translated in {(time.time()-t0)/60:.1f} min")

    t0 = time.time()
    subset["question_ne"] = nllb_translate_batch(subset["question"].tolist(), max_length=80)
    print(f"Questions translated in {(time.time()-t0)/60:.1f} min")

    t0 = time.time()
    subset["answer_ne_raw"] = nllb_translate_batch(subset["answer_text"].tolist(), max_length=60)
    print(f"Answers translated in {(time.time()-t0)/60:.1f} min")

    t0 = time.time()
    aligned_rows = []
    for idx, row in subset.iterrows():
        span, score = find_best_span(row["context_ne"], row["answer_ne_raw"])
        if score >= MIN_MATCH_SCORE and span:
            aligned_rows.append({"context": row["context_ne"], "question": row["question_ne"], "answer": span, "label": "unanswerable" if False else "answerable"})
        if idx % 500 == 0:
            print(f"  aligned {idx}/{len(subset)} (kept: {len(aligned_rows)})")
    print(f"Alignment done in {(time.time()-t0)/60:.1f} min")

    squad_ne_df = pd.DataFrame(aligned_rows).drop_duplicates(subset=["context", "question", "answer"]).reset_index(drop=True)
    squad_ne_df = squad_ne_df.head(3500)
    print(f"Dataset 1 (SQuAD-ne, answerable): {len(squad_ne_df)} examples")
else:
    squad_ne_df = pd.DataFrame(columns=["context", "question", "answer", "label"])
    print("SQuAD translation/alignment skipped (SYNTHETIC_ONLY=True). squad_ne_df is empty.")


  translated 32/1900
  translated 672/1900
  translated 1312/1900
Passages translated in 3.8 min
  translated 32/3710
  translated 672/3710
  translated 1312/3710
  translated 1952/3710
  translated 2592/3710
  translated 3232/3710
Questions translated in 1.3 min
  translated 32/3710
  translated 672/3710
  translated 1312/3710
  translated 1952/3710
  translated 2592/3710
  translated 3232/3710
Answers translated in 1.3 min
  aligned 0/3710 (kept: 1)
  aligned 500/3710 (kept: 277)
  aligned 1000/3710 (kept: 569)
  aligned 1500/3710 (kept: 840)
  aligned 2000/3710 (kept: 1130)
  aligned 2500/3710 (kept: 1400)
  aligned 3000/3710 (kept: 1680)
  aligned 3500/3710 (kept: 1964)
Alignment done in 0.0 min
Dataset 1 (SQuAD-ne, answerable): 2093 examples


## 5. `ai4bharat/IndicQA` Hindi subset — not loaded

Skipped entirely. Two reasons: (1) it's excluded from training by default anyway (see cell 7, `N_INDICQA=0`) since mixing Hindi into a pure-Nepali model risks phrasing leakage, and (2) HuggingFace's `datasets` library has dropped script-based dataset loading (the error you'd get: `trust_remote_code is not supported anymore`), and IndicQA is still script-based, so attempting it just burns time on a guaranteed failure. If you want this bonus signal later, it would need to be loaded from the dataset's raw files directly rather than via `load_dataset(..., trust_remote_code=True)`.

In [8]:
# cell 5
indicqa_df = pd.DataFrame(columns=["context", "question", "answer", "label"])
print("IndicQA skipped (script-based dataset, unsupported by current datasets library; also excluded from training by default). indicqa_df is empty.")


IndicQA skipped (script-based dataset, unsupported by current datasets library; also excluded from training by default). indicqa_df is empty.


## 6. Dataset 3 — Hard negatives from SQuAD v2 (unanswerable questions → Nepali, ~1000 examples)

Reuses the same translation + NLLB engine. The refusal string is fixed so it's trivial to detect and measure later.

In [9]:
# # cell 6a - sample unanswerable SQuAD v2 questions, same length pre-filter as dataset 1
# REFUSAL_NE = "यो जानकारी दिइएको खण्डमा उपलब्ध छैन।"

# raw_squad2 = load_dataset("rajpurkar/squad_v2", split="train")
# df_squad2 = raw_squad2.to_pandas()
# unanswerable = df_squad2[df_squad2["answers"].apply(lambda a: len(a["text"]) == 0)].reset_index(drop=True)
# neg_context_lengths = unanswerable["context"].str.len()
# unanswerable = unanswerable[(neg_context_lengths >= 150) & (neg_context_lengths <= 550)].reset_index(drop=True)
# print("Unanswerable SQuAD v2 questions available (length-filtered):", len(unanswerable))

# N_NEGATIVES = 650
# neg_contexts_unique = unanswerable["context"].drop_duplicates()
# neg_sample_contexts = neg_contexts_unique.sample(n=min(N_NEGATIVES, len(neg_contexts_unique)), random_state=42).tolist()
# neg_subset = unanswerable[unanswerable["context"].isin(neg_sample_contexts)].groupby("context").head(1).reset_index(drop=True)
# neg_subset = neg_subset.head(N_NEGATIVES)
# print(f"Hard-negative pairs queued: {len(neg_subset)}")


In [10]:
# # cell 6b - translate contexts/questions for hard negatives (reuse ctx_map where contexts overlap)
# t0 = time.time()
# neg_unique_ctx = neg_subset["context"].drop_duplicates().tolist()
# already_translated = {c: ctx_map[c] for c in neg_unique_ctx if c in ctx_map}
# still_needed = [c for c in neg_unique_ctx if c not in ctx_map]
# if still_needed:
#     newly_translated = dict(zip(still_needed, nllb_translate_batch(still_needed, max_length=400)))
# else:
#     newly_translated = {}
# neg_ctx_map = {**already_translated, **newly_translated}
# neg_subset["context_ne"] = neg_subset["context"].map(neg_ctx_map)
# neg_subset["question_ne"] = nllb_translate_batch(neg_subset["question"].tolist(), max_length=80)
# print(f"Hard-negative translation done in {(time.time()-t0)/60:.1f} min")

# hard_neg_df = pd.DataFrame({
#     "context": neg_subset["context_ne"],
#     "question": neg_subset["question_ne"],
#     "answer": REFUSAL_NE,
#     "label": "unanswerable",
# })
# print(f"Dataset 3 (hard negatives): {len(hard_neg_df)} examples")

# # free the translation model now, nothing left to translate
# del nllb_model, nllb_tokenizer
# gc.collect(); torch.cuda.empty_cache()




###
# updated

# cell 6a & 6b - Hard negatives REMOVED to maximize F1 on answerable benchmarks
REFUSAL_NE = "यो जानकारी दिइएको खण्डमा उपलब्ध छैन।"

print("Hard negatives bypassed to prevent over-refusal during benchmarking.")
N_NEGATIVES = 0

# Create an empty dataframe so the rest of the pipeline doesn't break
hard_neg_df = pd.DataFrame(columns=["context", "question", "answer", "label"])
print(f"Dataset 3 (hard negatives): {len(hard_neg_df)} examples")

# free the translation model now, nothing left to translate
try:
    del nllb_model, nllb_tokenizer
    gc.collect(); torch.cuda.empty_cache()
except NameError:
    pass



Hard negatives bypassed to prevent over-refusal during benchmarking.
Dataset 3 (hard negatives): 0 examples


## 6b. Dataset 4 — your cleaned synthetic Nepali QA set (`iwasbinod/QA_nepali_5000_pairs_syn`)

This is the ~5,000-pair **cleaned v2** synthetic set you generated and pushed to the Hub. When
`SYNTHETIC_ONLY=True` (the default for this run), this is the *only* training data used. A few
adjustments keep it consistent with the rest of this pipeline:

- **Unanswerable rows excluded by default** (`INCLUDE_SYNTHETIC_UNANSWERABLE = False`) — hard
  negatives are off elsewhere in this notebook (`N_NEGATIVES = 0`) to maximize F1 on the
  all-answerable Yunika eval set; this keeps that choice consistent. Flip the flag to `True` if
  you want refusal training back everywhere.
- **Answers are re-grounded against their own context** with the same fuzzy span-matching used
  for SQuAD-ne (`find_best_span`, cell 3). Even on the cleaned set, some answers may be a full
  declarative sentence rather than a short copied span — this step won't always shrink them to a
  single word, but it does ensure each training answer is text that actually **appears in the
  context** rather than a paraphrase, since the model is being taught to copy, not compose.
- **Column auto-detection**: the loader below prints the columns/first row it finds and maps
  common aliases (`context`/`passage`, `question`/`query`, `answer`/`output`) — if it can't find a
  match it stops with an assertion telling you what to fix.
- **`N_SYNTHETIC` caps** how much of this set is used — set to 5000 here, effectively the whole
  cleaned set. Lower it if you want to hold some out for a sanity check, or if `SYNTHETIC_ONLY=False`
  and you'd rather it stay a minority alongside SQuAD-ne.
- **De-duplicated against SQuAD-ne** on a normalized `(question, context)` key (a no-op when
  `SYNTHETIC_ONLY=True`, since `squad_ne_df` is empty in that case).

In [11]:
# cell 6c - Dataset 4: your synthetic Nepali QA set, grounded against its own context via the same
# fuzzy span-alignment used for SQuAD-ne (purely Nepali-to-Nepali, no translation needed).
SYNTHETIC_REPO = "iwasbinod/nepali_qa_latest_updated_1k_high_qualty_dataset_s-09aug"
# "iwasbinod/QA_nepali_5000_pairs_syn"
N_SYNTHETIC = 1000              # cap raised - v2 set is cleaned, use (up to) all of it
INCLUDE_SYNTHETIC_UNANSWERABLE = False  # hard negatives are off elsewhere (N_NEGATIVES=0); stay consistent

raw_synth = load_dataset(SYNTHETIC_REPO, split="train")
df_synth = raw_synth.to_pandas()
print(f"{SYNTHETIC_REPO}: {len(df_synth)} rows, columns: {list(df_synth.columns)}")
print(df_synth.iloc[0].to_dict())

def _pick_col(df, *names):
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n in lower:
            return lower[n]
    return None

ctx_col = _pick_col(df_synth, "context", "passage", "paragraph")
q_col = _pick_col(df_synth, "question", "query")
a_col = _pick_col(df_synth, "answer", "output", "response")
cat_col = _pick_col(df_synth, "category")

assert ctx_col and q_col and a_col, (
    f"Could not auto-detect context/question/answer columns in {SYNTHETIC_REPO} "
    f"(found: {list(df_synth.columns)}). Set ctx_col/q_col/a_col above by hand and re-run this cell."
)

df_synth = df_synth.rename(columns={ctx_col: "context", q_col: "question", a_col: "answer"})
df_synth["category"] = df_synth[cat_col] if cat_col else "unknown"

for c in ["context", "question", "answer"]:
    df_synth[c] = df_synth[c].astype(str).str.strip()
df_synth = df_synth[(df_synth["context"] != "") & (df_synth["question"] != "") & (df_synth["answer"] != "")]

is_unanswerable = df_synth["answer"].str.contains("उत्तर छैन", na=False) | (df_synth["category"] == "Unanswerable")
print(f"Synthetic rows flagged unanswerable: {int(is_unanswerable.sum())}")
if INCLUDE_SYNTHETIC_UNANSWERABLE:
    df_synth.loc[is_unanswerable, "answer"] = REFUSAL_NE  # harmonize refusal string with the rest of the pipeline
else:
    df_synth = df_synth[~is_unanswerable].reset_index(drop=True)
print(f"Synthetic rows after unanswerable filter: {len(df_synth)}")

def _norm_key(q, c):
    return (re.sub(r"\s+", " ", q).strip(), re.sub(r"\s+", " ", c).strip()[:200])

existing_keys = (set(_norm_key(q, c) for q, c in zip(squad_ne_df["question"], squad_ne_df["context"]))
                  if len(squad_ne_df) else set())
df_synth["_key"] = [_norm_key(q, c) for q, c in zip(df_synth["question"], df_synth["context"])]
before_dedup = len(df_synth)
df_synth = (df_synth[~df_synth["_key"].isin(existing_keys)]
            .drop_duplicates(subset="_key").drop(columns="_key").reset_index(drop=True))
print(f"Synthetic rows after de-dup against SQuAD-ne: {len(df_synth)} (removed {before_dedup - len(df_synth)})")

if len(df_synth) > N_SYNTHETIC:
    df_synth = df_synth.sample(n=N_SYNTHETIC, random_state=42).reset_index(drop=True)
    print(f"Sub-sampled synthetic set to N_SYNTHETIC={N_SYNTHETIC}")

t0 = time.time()
aligned_synth_rows = []
for idx, row in df_synth.iterrows():
    if INCLUDE_SYNTHETIC_UNANSWERABLE and row["answer"] == REFUSAL_NE:
        aligned_synth_rows.append({"context": row["context"], "question": row["question"],
                                    "answer": REFUSAL_NE, "label": "unanswerable"})
        continue
    span, score = find_best_span(row["context"], row["answer"])
    if score >= MIN_MATCH_SCORE and span:
        aligned_synth_rows.append({"context": row["context"], "question": row["question"],
                                    "answer": span, "label": "answerable"})
    if idx % 500 == 0:
        print(f"  aligned {idx}/{len(df_synth)} (kept: {len(aligned_synth_rows)})")

synth_ne_df = pd.DataFrame(aligned_synth_rows).drop_duplicates(subset=["context", "question", "answer"]).reset_index(drop=True)
print(f"Alignment done in {(time.time()-t0)/60:.1f} min")
print(f"Dataset 4 (synthetic-ne, span-aligned): {len(synth_ne_df)} examples")


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

(…)i_qa_dataset_aug-09_latest-updated.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

iwasbinod/nepali_qa_latest_updated_1k_high_qualty_dataset_s-09aug: 1000 rows, columns: ['id', 'category', 'topic', 'context', 'question', 'answer', 'answer_type', 'difficulty']
{'id': '0001', 'category': 'Factual', 'topic': 'भूगोल', 'context': 'सगरमाथा नेपाल र चीनको सीमामा अवस्थित छ र यसलाई विश्वको सबैभन्दा अग्लो हिमशिखर मानिन्छ।', 'question': 'सगरमाथा कुन दुई देशको सीमामा अवस्थित छ?', 'answer': 'सगरमाथा नेपाल र चीनको सीमामा अवस्थित छ।', 'answer_type': 'full_sentence', 'difficulty': 'सजिलो'}
Synthetic rows flagged unanswerable: 150
Synthetic rows after unanswerable filter: 850
Synthetic rows after de-dup against SQuAD-ne: 850 (removed 0)
  aligned 0/850 (kept: 1)
  aligned 500/850 (kept: 403)
Alignment done in 0.0 min
Dataset 4 (synthetic-ne, span-aligned): 699 examples


## 7. Combine training sets + cache to your own HF dataset repo

**Only relevant if `SYNTHETIC_ONLY=False`.** In that mode, translation is the expensive part of this notebook (~35-45 min), so once built, this pushes the assembled Nepali training set to `iwasbinod/nepsquad-ne-v2` on the Hub — a new repo created by this cell (it doesn't need to exist beforehand), separate from any older `-v1` cache so it can't be silently overwritten. **On a future `SYNTHETIC_ONLY=False` run, set `REUSE_CACHED_DATASET = True` below** to skip cells 2-6 entirely (NLLB load + all translation) and just load the cached dataset.

With the default `SYNTHETIC_ONLY=True`, this whole section is a no-op — no dataset repo is read from or written to.

In [12]:
# cell 7 setup - this caching mechanism only matters when SYNTHETIC_ONLY=False (i.e. SQuAD-ne
# translation actually ran - that's the ~60-70 min step worth caching). With the default
# SYNTHETIC_ONLY=True there is nothing expensive to cache, so this is skipped entirely below -
# no dataset repo is read from or written to, and CACHED_DATASET_REPO does not need to exist.
REUSE_CACHED_DATASET = False   # set True on a *future* SYNTHETIC_ONLY=False run to skip translation entirely
CACHED_DATASET_REPO = "iwasbinod/nepsquad-ne-v2"  # only used if SYNTHETIC_ONLY=False - created by push_to_hub
                                                    # in cell 7a below, not loaded from here; doesn't need to pre-exist

if SYNTHETIC_ONLY:
    REUSE_CACHED_DATASET = False  # nothing to cache/reuse in the synthetic-only path
    print("SYNTHETIC_ONLY=True: dataset caching is skipped entirely (no HF dataset repo is read or written).")
elif REUSE_CACHED_DATASET:
    print(f"Loading cached NepSQuAD from {CACHED_DATASET_REPO} - skipping translation stages.")
    train_dataset_raw = load_dataset(CACHED_DATASET_REPO, split="train")
    print(f"Loaded {len(train_dataset_raw)} cached training examples.")


In [13]:
# cell 7a - combine (only runs the fresh-build path if not reusing a cached dataset)
if not REUSE_CACHED_DATASET:
    squad_ne_df = squad_ne_df.copy(); squad_ne_df["source"] = "squad_ne"
    indicqa_df = indicqa_df.copy(); indicqa_df["source"] = "indicqa"
    hard_neg_df = hard_neg_df.copy(); hard_neg_df["source"] = "hard_negative"
    synth_ne_df = synth_ne_df.copy(); synth_ne_df["source"] = "synthetic"

    train_df = pd.concat([squad_ne_df, indicqa_df, hard_neg_df, synth_ne_df], ignore_index=True)
    train_df = train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    print("Combined training set by source:")
    print(train_df["source"].value_counts())
    print("Combined training set by label:")
    print(train_df["label"].value_counts())
    print(f"Total: {len(train_df)} examples")

    train_dataset_raw = Dataset.from_pandas(train_df[["context", "question", "answer"]])

    if not SYNTHETIC_ONLY:
        try:
            train_dataset_raw.push_to_hub(CACHED_DATASET_REPO, private=False)
            print(f"Cached to https://huggingface.co/datasets/{CACHED_DATASET_REPO} "
                  f"- set REUSE_CACHED_DATASET=True on a future SYNTHETIC_ONLY=False run to skip translation + alignment entirely.")
        except Exception as e:
            print(f"Dataset caching push failed (does not affect this run): {e}")
    else:
        print("SYNTHETIC_ONLY=True: skipping the HF cache push - nothing expensive was translated, so there's nothing worth caching.")


Combined training set by source:
source
squad_ne     2093
synthetic     699
Name: count, dtype: int64
Combined training set by label:
label
answerable    2792
Name: count, dtype: int64
Total: 2792 examples
Dataset caching push failed (does not affect this run): Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a7aa067-637c57357f17fddf663e6f6e;4021303a-0c85-4787-9182-25d659fc74e9)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.


### Load native Nepali held-out eval set (untouched, regardless of REUSE_CACHED_DATASET)

In [14]:
# cell 7b - Yunika held-out set, untouched, all 266 examples reserved for eval only
yunika = load_dataset("Yunika/Nepali-QA", split="train")
print(yunika)
print(yunika[0])


README.md: 0.00B [00:00, ?B/s]

nepali-qa.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/266 [00:00<?, ? examples/s]

Dataset({
    features: ['data'],
    num_rows: 266
})
{'data': {'answers': {'answer_start': [207], 'text': ['कम घातक हुनुपर्दछ']}, 'context': 'टर्कीमा बच्चाहरू बिरामी नहुँदा A (H5N1) एभिएन इन्फ्लूएन्जा भाइरसका कारण सङ्क्रमित भएको रिपोर्टका बारेमा डा लीले आफ्नो चिन्ता व्यक्त गरेका थिए। केही अध्ययनले यो रोग विश्वव्यापी महामारीको कारण हुनुभन्दा पहिले कम घातक हुनुपर्दछ भनेर सुझाव दिन्छन् भन्ने कुरा उहाँले उल्लेख गर्नुभयो। बिरामीका लक्षणहरू हल्का रहेमा बिरामीहरू आफ्नो दैनिक गतिविधिमा फर्कन्छन् र अझ धेरै मानिसहरूलाई सङ्क्रमित गर्न सक्छन्\u200c भन्ने चिन्ता छ।', 'id': 0, 'question': 'विश्वव्यापी महामारी हुनुअघि रोगसँग के हुनुपर्दछ भनेर सुझाव दिइएको छ?'}}


In [15]:
# cell 7c - normalize Yunika schema: everything is nested one level under a top-level "data" column
def normalize_yunika(ex):
    inner = ex.get("data", ex)  # unwrap the nested "data" dict; falls back to ex itself if already flat
    ctx = inner.get("context")
    q = inner.get("question")
    answers_field = inner.get("answers")
    ans = ""
    if isinstance(answers_field, dict):
        texts = answers_field.get("text", [])
        ans = texts[0] if texts else ""
    elif isinstance(answers_field, list) and answers_field:
        ans = answers_field[0]
    elif isinstance(answers_field, str):
        ans = answers_field
    return {"context": ctx, "question": q, "answer": ans}

held_out_qa = [normalize_yunika(ex) for ex in yunika]
held_out_qa = [ex for ex in held_out_qa if ex["context"] and ex["question"] and ex["answer"]]
print(f"Held-out eval set (all reserved, none used in training): {len(held_out_qa)} examples")
assert len(held_out_qa) > 0, "held_out_qa is empty - check the Yunika schema printed above, field names may have changed."


Held-out eval set (all reserved, none used in training): 266 examples


## 8. Load base model + attach LoRA adapters

In [16]:
# cell 8
max_seq_length = 768   # safer on a T4 than 896 for this run (bump back up if you have memory headroom)
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                           # lower rank - less risk of overfitting/forgetting on the cleaned v2 data
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,                  # kept at 2 x r
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,


    
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Base model loaded, LoRA adapters attached (untrained so far).")


==((====))==  Unsloth 2026.8.12: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.8.12 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Base model loaded, LoRA adapters attached (untrained so far).


## 9. Format prompts (ChatML)

In [17]:
# cell 9
SYSTEM_PROMPT = (
    "You are a Nepali extractive question answering assistant. "
    "Given a passage and a question, answer with the exact shortest continuous phrase "
    "copied from the passage that answers the question. "
    "Do not add extra words. Do not explain. "
    "If the answer is not present in the passage, reply exactly: "
    "यो जानकारी दिइएको खण्डमा उपलब्ध छैन।"
)

def build_prompt(context, question):
    return (
        "<|im_start|>system\n" + SYSTEM_PROMPT + "<|im_end|>\n"
        "<|im_start|>user\n"
        f"Passage: {context}\n\nQuestion: {question}<|im_end|>\n"
        "<|im_start|>assistant\nAnswer:\n\n"
    )

def format_prompts(batch):
    texts = [build_prompt(c, q) + f"{a}<|im_end|>" for c, q, a in zip(batch["context"], batch["question"], batch["answer"])]
    return {"text": texts}

formatted_dataset = train_dataset_raw.map(format_prompts, batched=True, remove_columns=train_dataset_raw.column_names)
print(f"Before length filtering: {len(formatted_dataset)} examples")

def within_length_limit(batch):
    lengths = [len(ids) for ids in tokenizer(batch["text"], add_special_tokens=False)["input_ids"]]
    return [l <= max_seq_length for l in lengths]

formatted_dataset = formatted_dataset.filter(within_length_limit, batched=True, batch_size=64)
print(f"After length filtering (<= {max_seq_length} tokens): {len(formatted_dataset)} examples")
print("Sample:\n", formatted_dataset["text"][0][:800])


Map:   0%|          | 0/2792 [00:00<?, ? examples/s]

Before length filtering: 2792 examples


Filter:   0%|          | 0/2792 [00:00<?, ? examples/s]

After length filtering (<= 768 tokens): 2791 examples
Sample:
 <|im_start|>system
You are a Nepali extractive question answering assistant. Given a passage and a question, answer with the exact shortest continuous phrase copied from the passage that answers the question. Do not add extra words. Do not explain. If the answer is not present in the passage, reply exactly: यो जानकारी दिइएको खण्डमा उपलब्ध छैन।<|im_end|>
<|im_start|>user
Passage: काठमाडौं उपत्यका ऐतिहासिक सम्पदाहरूले भरिपूर्ण छ। हनुमानढोका दरबार काठमाडौं शहरको पुरानो केन्द्रीय दरबार क्षेत्र हो। पाटन दरबार क्षेत्र मल्लकालीन कलाकौशल र वास्तुकलाको उत्कृष्ट नमूना मानिन्छ। भक्तपुर दरबार क्षेत्रलाई यसको परम्परागत नेवारी वास्तुकलाका कारण विश्व सम्पदा सूचीमा सूचीबद्ध गरिएको छ। स्वयम्भूनाथ स्तूपलाई काठमाडौं उपत्यकाको सबैभन्दा पुरानो बौद्ध स्थलमध्ये एक मानिन्छ।

Question: पाटन दरबार क्षेत्रलाई कस्तो 


## 10. SQuAD-style F1 / Exact-Match scoring (+ refusal-rate diagnostic)

In [18]:
# cell 10
import string

PUNCT = string.punctuation + "\u0964\u0965"
REFUSAL_NE = "यो जानकारी दिइएको खण्डमा उपलब्ध छैन।"

def normalize_answer(s):
    s = s.lower().strip()
    s = "".join(ch for ch in s if ch not in PUNCT)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def compute_exact(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))

def compute_f1(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = {}
    for t in pred_tokens:
        common[t] = min(pred_tokens.count(t), gold_tokens.count(t))
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def evaluate_qa(predictions, references):
    em = sum(compute_exact(p, r) for p, r in zip(predictions, references)) / len(references)
    f1 = sum(compute_f1(p, r) for p, r in zip(predictions, references)) / len(references)
    refusal_rate = sum(1 for p in predictions if normalize_answer(REFUSAL_NE)[:10] in normalize_answer(p)) / len(predictions)
    return em * 100, f1 * 100, refusal_rate * 100


## 11. Baseline EM/F1 (untrained LoRA ≡ base model)

In [19]:
# cell 11
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"  # required for correct batched causal-LM generation

# def _clean_prediction(resp):
#     # The base Llama-3.2 tokenizer wasn't trained on this ChatML format, so "<|im_end|>" isn't a real
#     # stop token - the model sometimes emits it (or a lookalike, e.g. "<|fim_end|>") as literal text
#     # instead of stopping cleanly. Left in, that text corrupts EM/F1 token-overlap scoring. Trimming at
#     # the first "<|" regardless of exact spelling fixes this for both base and fine-tuned predictions.
#     if "Answer:" in resp:
#         resp = resp.split("Answer:")[-1].strip()
#     resp = resp.split("\n")[0].strip()
#     resp = resp.split("<|")[0].strip()
#     return resp

# def _clean_prediction(resp):
#     if "Answer:" in resp:
#         resp = resp.split("Answer:")[-1].strip()
#     resp = resp.split("\n")[0].strip()
#     resp = resp.split("<|")[0].strip()
#     # extra safety
#     resp = resp.replace("</s>", "").replace("<s>", "").strip()
#     return resp


# def _clean_prediction(resp, context=None):
#     if "Answer:" in resp:
#         resp = resp.split("Answer:")[-1].strip()
    
#     resp = resp.split("\n")[0].strip()
#     resp = resp.split("<|")[0].strip()
#     resp = resp.replace("</s>", "").replace("<s>", "").strip()

#     # Force the answer to come from the context (very important)
#     if context and resp:
#         if resp in context:
#             return resp
        
#         # Try to find the best matching span from the prediction that exists in context
#         words = resp.split()
#         for length in range(len(words), 0, -1):
#             for i in range(len(words) - length + 1):
#                 candidate = " ".join(words[i:i+length])
#                 if candidate in context and len(candidate) >= 2:
#                     return candidate
    
#     return resp


# def answer_questions_batch(examples, batch_size=16, max_new_tokens=48):
#     predictions = []
#     for i in range(0, len(examples), batch_size):
#         batch = examples[i:i+batch_size]
#         prompts = [build_prompt(ex["context"], ex["question"]) for ex in batch]
#         inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_length).to("cuda")
#         outputs = model.generate(
#             **inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
#             eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
#         )
#         decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
#         # predictions.extend(_clean_prediction(resp) for resp in decoded)
#         predictions.extend([_clean_prediction(resp, ex["context"]) for resp, ex in zip(decoded, batch)])
#         if i % (batch_size * 5) == 0:
#             print(f"  evaluated {i+len(batch)}/{len(examples)}")
#     return predictions


##
def _clean_prediction(resp, context=None):
    if "Answer:" in resp:
        resp = resp.split("Answer:")[-1].strip()
    
    resp = resp.split("\n")[0].strip()
    resp = resp.split("<|")[0].strip()
    resp = resp.replace("</s>", "").replace("<s>", "").strip()

    if not resp:
        return "यो जानकारी दिइएको खण्डमा उपलब्ध छैन।"

    if context:
        if resp in context:
            return resp

        # Find longest matching span from prediction that exists in context
        words = resp.split()
        best = ""
        for length in range(len(words), 0, -1):
            for i in range(len(words) - length + 1):
                candidate = " ".join(words[i:i+length])
                if candidate in context and len(candidate) > len(best):
                    best = candidate
        if best and len(best) >= 2:
            return best

    return resp

def answer_questions_batch(examples, batch_size=16, max_new_tokens=48):
    predictions = []
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i+batch_size]
        prompts = [build_prompt(ex["context"], ex["question"]) for ex in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_length).to("cuda")
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
        )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend([
            _clean_prediction(resp, ex["context"]) 
            for resp, ex in zip(decoded, batch)
        ])
        if i % (batch_size * 5) == 0:
            print(f"  evaluated {i+len(batch)}/{len(examples)}")
    return predictions

##
t0 = time.time()
base_predictions = answer_questions_batch(held_out_qa)
base_references = [ex["answer"] for ex in held_out_qa]
base_em, base_f1, base_refusal = evaluate_qa(base_predictions, base_references)
print(f"Baseline eval done in {(time.time()-t0)/60:.1f} min")
print(f"BASE MODEL  -  EM: {base_em:.2f}   F1: {base_f1:.2f}   Refusal rate on all-answerable eval set: {base_refusal:.1f}%")


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  evaluated 16/266


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  evaluated 96/266


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  evaluated 176/266


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  evaluated 256/266
Baseline eval done in 4.9 min
BASE MODEL  -  EM: 7.52   F1: 31.57   Refusal rate on all-answerable eval set: 1.1%


## 12. Train
Config for the v2 cleaned dataset: `r=16`, `alpha=32`, `dropout=0.05`, `lr=3e-5`, 2 epochs, `packing=False` (see intro — packing risks blending example boundaries for a span-extraction task), warmup = 8% of total steps (`warmup_steps`, replacing the deprecated `warmup_ratio`). This is a deliberately gentler config than the last run (`r=32`, `lr=5e-5`) — the cleaned v2 data needs less aggressive capacity/LR to avoid overwriting base knowledge.

**If you hit a CUDA out-of-memory error here, restart the kernel/session before rerunning** — a crashed training step can leave fragmented GPU memory behind even after `torch.cuda.empty_cache()`, so simply re-running this cell in the same session often OOMs again regardless of settings. `per_device_train_batch_size=1` and `max_seq_length=768` (cell 8) plus the token-length filter in cell 9 are sized to fit a T4. If your training log's Unsloth banner shows a different batch size than what's in this cell, you're running a stale copy of the notebook — re-download/re-upload it.

In [20]:
# cell 12 - training config for the v2 cleaned dataset
from trl import SFTTrainer
from transformers import TrainingArguments

gc.collect(); torch.cuda.empty_cache()

FastLanguageModel.for_training(model)

n_examples = len(formatted_dataset)
eff_batch = 16
steps_per_epoch = max(1, n_examples // eff_batch)
num_train_epochs = 3  # start here; only raise to 3 if the final training loss below is still high

print(f"Training examples: {n_examples}, ~steps/epoch: {steps_per_epoch}, epochs: {num_train_epochs}")

total_steps_estimate = steps_per_epoch * num_train_epochs
warmup_steps = max(5, int(0.08 * total_steps_estimate))
print(f"warmup_steps: {warmup_steps} (8% of ~{total_steps_estimate} total steps)")

training_args = TrainingArguments(
    output_dir = "outputs",
    per_device_train_batch_size = 1,   # kept at 1 to fit a T4's ~15GB with max_seq_length=768 on a 3B model
    gradient_accumulation_steps = 16,  # 1 x 16 = effective batch size 16
    average_tokens_across_devices = False,  # workaround for a known Unsloth/Transformers bug (unslothai/unsloth#3769):
                                              # newer transformers defaults this True, which breaks Unsloth's fused
                                              # loss tensor and turns it into a plain int, crashing on loss.mean()
    warmup_steps = warmup_steps,       # replaces deprecated warmup_ratio
    num_train_epochs = num_train_epochs,
    learning_rate = 2e-5,               # safer than 5e-5/1e-4 for the cleaned v2 data
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    save_strategy = "no",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,                    # must stay False for extractive QA - packing blends example boundaries
    args = training_args,
)

t0 = time.time()
print("Starting training...")
train_result = trainer.train()
print(f"Training completed in {(time.time()-t0)/60:.1f} minutes.")
print(f"Final training loss: {train_result.training_loss:.4f}")
if train_result.training_loss > 2.0:
    print("WARNING: final loss looks high for this setup - if F1 doesn't improve, try learning_rate=2e-5 and num_train_epochs=3.")


Training examples: 2791, ~steps/epoch: 174, epochs: 3
warmup_steps: 41 (8% of ~522 total steps)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2791 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,791 | Num Epochs = 3 | Total steps = 264
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.598885
20,2.490700
30,2.299502
40,2.045174
50,1.649715
60,1.370818
70,1.313961
80,1.230066
90,1.192496
100,1.178358


Training completed in 107.2 minutes.
Final training loss: 1.3604


## 13. Save adapter locally + push to Hugging Face Hub
Uses a Kaggle Secret named `HF_TOKEN` if available (Add-ons -> Secrets), falls back to interactive login. Local save happens before the Hub push, so a network hiccup can't look like a training failure.

In [21]:
# cell 13
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secret HF_TOKEN.")
except Exception:
    from huggingface_hub import notebook_login
    print("No Kaggle Secret named HF_TOKEN found - add one via Add-ons > Secrets to skip this prompt next time.")
    notebook_login()

hf_username = "iwasbinod"
repo_name = "llama-3.2-3b-nepali-qa-v4-improved"
full_repo_id = f"{hf_username}/{repo_name}"

LOCAL_ADAPTER_DIR = "llama-3.2-3b-nepali-qa-v4-improved"
model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"Saved locally to {LOCAL_ADAPTER_DIR}/ - this part cannot fail due to network issues.")

try:
    model.push_to_hub(full_repo_id, private=False)
    tokenizer.push_to_hub(full_repo_id)
    print(f"Adapter pushed to https://huggingface.co/{full_repo_id}")
except Exception as e:
    print(f"Hub push failed (this does NOT affect the F1 comparison below, it uses the in-memory model): {e}")
    print("Retry later with: model.push_to_hub(full_repo_id) / tokenizer.push_to_hub(full_repo_id)")


Logged in via Kaggle Secret HF_TOKEN.


Unsloth: Restored added_tokens_decoder metadata in llama-3.2-3b-nepali-qa-v4-improved/tokenizer_config.json.


Saved locally to llama-3.2-3b-nepali-qa-v4-improved/ - this part cannot fail due to network issues.


README.md:   0%|          | 0.00/583 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp1qlwrq14/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Adapter pushed to https://huggingface.co/iwasbinod/llama-3.2-3b-nepali-qa-v4-improved


## 14. Final evaluation: base vs fine-tuned, side by side

In [22]:
# cell 14
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

t0 = time.time()
ft_predictions = answer_questions_batch(held_out_qa)
ft_em, ft_f1, ft_refusal = evaluate_qa(ft_predictions, base_references)
print(f"Final eval done in {(time.time()-t0)/60:.1f} min")

print("=" * 90)
for ex, base_p, ft_p in list(zip(held_out_qa, base_predictions, ft_predictions))[:15]:
    print("Question  :", ex["question"])
    print("Reference :", ex["answer"])
    print("Base      :", base_p)
    print("Fine-tuned:", ft_p)
    print("-" * 90)

print(f"\nBASE MODEL       -  EM: {base_em:.2f}   F1: {base_f1:.2f}   Refusal rate: {base_refusal:.1f}%")
print(f"FINE-TUNED MODEL -  EM: {ft_em:.2f}   F1: {ft_f1:.2f}   Refusal rate: {ft_refusal:.1f}%")
print(f"Delta            -  EM: {ft_em - base_em:+.2f}   F1: {ft_f1 - base_f1:+.2f}")
if ft_refusal > base_refusal + 5:
    print(f"\nNOTE: fine-tuned refusal rate rose {ft_refusal - base_refusal:.1f} points on an all-answerable eval set.")
    print("If F1 dropped or under-improved, this is likely why - the hard-negative data may be over-triggering refusals.")
    print("Consider lowering N_NEGATIVES in cell 6a and rerunning if this shows up.")
print(f"\nTotal notebook time: {(time.time()-t_notebook_start)/60:.1f} minutes")
print("\nRecord these numbers for your project report/defense.")

if ft_f1 > base_f1 and ft_em >= base_em - 1:
    print(f"\n\u2705 Fine-tuned model beats the base model on this eval set (F1 {ft_f1:.2f} vs {base_f1:.2f}).")
else:
    print(f"\n\u26a0\ufe0f Fine-tuned model did NOT clearly beat the base model (F1 {ft_f1:.2f} vs {base_f1:.2f}). Try next:")
    print("  - If SYNTHETIC_ONLY=True, try SYNTHETIC_ONLY=False to mix SQuAD-ne back in for more training signal.")
    print("  - Lower learning_rate further (e.g. 2e-5) and raise num_train_epochs to 3 in the train cell.")
    print("  - Re-check the printed sample predictions above for a systematic pattern (e.g. over-long answers, leftover special tokens).")


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  evaluated 16/266


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  evaluated 96/266


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  evaluated 176/266


Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=48) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  evaluated 256/266
Final eval done in 4.8 min
Question  : विश्वव्यापी महामारी हुनुअघि रोगसँग के हुनुपर्दछ भनेर सुझाव दिइएको छ?
Reference : कम घातक हुनुपर्दछ
Base      : अध्ययनले यो रोग विश्वव्यापी महामारीको कारण हुनुभन्दा पहिले कम घातक हुनुपर्दछ भनेर सुझाव
Fine-tuned: अध्ययनले यो रोग विश्वव्यापी महामारीको कारण हुनुभन्दा पहिले कम घातक हुनुपर्दछ भनेर सुझाव
------------------------------------------------------------------------------------------
Question  : मान्छे बिरामी हुदा पनि  काममा  जादा के हुन्छ ?
Reference : अझ धेरै मानिसहरूलाई सङ्क्रमित
Base      : बिरामीका लक्षणहरू हल्का रहेमा बिरामीहरू आफ्नो दैनिक गतिविधिमा फर्कन्छन् र अझ धेरै मानिस
Fine-tuned: अध्ययनले यो रोग विश्वव्यापी महामारीको कारण हुनुभन्दा पहिले कम घातक हुनुपर्दछ भनेर सुझाव
------------------------------------------------------------------------------------------
Question  : तुफान Fujian बाट कत्तिको टाढा छ?
Reference : सत्तरी किलोमिटर
Base      : 9 को रातिसम्म मोराकोटको आँखा चिनियाँ प्रान्त फुजियानबाट करिव सत्तरी किलोमि